# Assignment 4 

# Task 1: Basketball Game Commentary Analysis

In [21]:
!pip install faster-whisper chromadb ollama sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [22]:
import os
from faster_whisper import WhisperModel
import chromadb
import ollama
from sentence_transformers import SentenceTransformer
import subprocess
import json
from typing import List, Dict

## Step 3: Load and Extract Audio Portion

We'll load the MP3 file from the video folder and extract a 20-30 minute portion using ffmpeg.

In [23]:
## This function takes the input path of the entire mp3 file, and creates a new mp3 file that is a fraction of the duration (I didn't want to embed the entire hour and a half long game)
def extract_audio_portion(
    input_path: str, 
    output_path: str, 
    start_minutes: int = 0, 
    duration_minutes: int = 25
) -> str:
    ## Convert to seconds for ffmpeg
    start_seconds = start_minutes * 60
    duration_seconds = duration_minutes * 60
    
    ## Build ffmpeg command - re-encode to MP3 since source is AAC in MP4 container
    command = [
        'ffmpeg',
        '-y', ## Overwrite output file if exists
        '-i', input_path,
        '-ss', str(start_seconds),
        '-t', str(duration_seconds),
        '-acodec', 'libmp3lame', ## Re-encode to MP3
        '-q:a', '2',
        output_path
    ]
    
    print(f"Extracting {duration_minutes} minutes starting at {start_minutes} min...")
    result = subprocess.run(command, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ffmpeg error: {result.stderr}")
        raise RuntimeError("Failed to extract audio portion")
    
    print(f"Extracted audio saved to: {output_path}")
    return output_path

In [24]:
## Path to the source MP3 file
SOURCE_AUDIO = "video/video.mp3"

## Extract 25-minute portion of the mp3 file
audio_path = extract_audio_portion(
    SOURCE_AUDIO, 
    "game_portion.mp3", 
    start_minutes=0,
    duration_minutes=25
)

print(f"\nUsing audio file: {audio_path}")

Extracting 25 minutes starting at 0 min...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Extracted audio saved to: game_portion.mp3

Using audio file: game_portion.mp3


In [25]:
## This function transcribes the audio file and adds timestamps with them 
def transcribe_with_timestamps(audio_path: str, model_size: str = "small") -> Dict:

    print(f"Loading Faster-Whisper model: {model_size}")
    
    ## Specification of whisper model that is being used
    model = WhisperModel(model_size, device="cpu", compute_type="int8")
    
    print(f"Transcribing: {audio_path}")
    print("This may take several minutes...")
    
    ## Transcribe with word timestamps
    segments, info = model.transcribe(
        audio_path,
        word_timestamps=True,
        vad_filter=True ## Filter out silence
    )
    
    print(f"Detected language: {info.language} (probability: {info.language_probability:.2f})")
    
    ## Convert generator to list and format
    segments_list = []
    full_text = []
    
    for segment in segments:
        segments_list.append({
            'id': segment.id,
            'start': segment.start,
            'end': segment.end,
            'text': segment.text.strip()
        })
        full_text.append(segment.text)
        print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text.strip()}")
    
    print(f"\nTranscription complete. Found {len(segments_list)} segments.")
    
    return {
        'text': ' '.join(full_text),
        'segments': segments_list,
        'language': info.language
    }

## This function changes the format of the timestamps to be in minutes and seconds and takes time in seconds as input
def format_timestamp(seconds: float) -> str:

    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"

## This function processes the transcription segments into structured chucks so they are ready for the embedding process (these have the timestamps with them)
def process_segments(transcription: Dict) -> List[Dict]:
 
    processed = []
    
    for segment in transcription['segments']:
        processed.append({
            'id': segment['id'],
            'start': segment['start'],
            'end': segment['end'],
            'start_formatted': format_timestamp(segment['start']),
            'end_formatted': format_timestamp(segment['end']),
            'text': segment['text']
        })
    
    return processed

In [26]:
## using a different model_size would increase the accuracy, but it would take a really long time (I have 8GB of RAM)
transcription = transcribe_with_timestamps(audio_path, model_size="small")

## Process segments
segments = process_segments(transcription)

## Make sure that the segments are structured correctly.
print("\nFirst 5 segments:")
for seg in segments[:5]:
    print(f"[{seg['start_formatted']} - {seg['end_formatted']}]: {seg['text']}")

Loading Faster-Whisper model: small
Transcribing: game_portion.mp3
This may take several minutes...
Detected language: en (probability: 0.99)
[0.02s -> 1.30s] Well, it's not breaking news.
[1.48s -> 2.86s] They're a different basketball team
[2.86s -> 4.50s] with a healthy hold, LeBron James.
[5.70s -> 8.08s] 15 games over 500 when he's healthy.
[8.54s -> 9.90s] Three games under 500.
[10.46s -> 12.70s] Creakes offense for himself and his teammates.
[13.38s -> 13.86s] They need him.
[14.46s -> 16.34s] They all, Jeff Steph Curry, what he's
[16.34s -> 17.98s] done over the past month and a half
[17.98s -> 19.58s] has been absolutely brilliant.
[20.26s -> 22.62s] He's as valuable as any player in this league.
[23.06s -> 25.02s] They play great defense, and they
[25.02s -> 26.94s] have the ultimate home run hitter.
[27.64s -> 30.52s] Well, beneath the team, it hasn't been easy this season,
[30.90s -> 33.62s] but both finished up the regular season strong.
[33.92s -> 36.02s] The Lakers ende

In [27]:
class CommentaryVectorStore:

    ## Initialization of the vector storage class
    def __init__(self, collection_name: str = "basketball_commentary"):

        ## Initialize embedding model
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        ## Initialize ChromaDB
        self.client = chromadb.Client()
        
        ## Delete existing collection if it exists
        try:
            self.client.delete_collection(collection_name)
        except:
            pass
        
        self.collection = self.client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        print(f"Initialized vector store: {collection_name}")
    
    ## Add the transcription segments into the vectore storage.
    def add_segments(self, segments: List[Dict]) -> None:

        ## Group segments into larger chunks for better context
        chunks = self._create_chunks(segments, chunk_size=5)
        
        documents = []
        embeddings = []
        metadatas = []
        ids = []
        
        for i, chunk in enumerate(chunks):
            ## Combine text from segments in chunk
            text = " ".join([seg['text'] for seg in chunk])
            
            ## Get embedding
            embedding = self.embedding_model.encode(text).tolist()
            
            ## Metadata with timestamps
            metadata = {
                'start_time': chunk[0]['start'],
                'end_time': chunk[-1]['end'],
                'start_formatted': chunk[0]['start_formatted'],
                'end_formatted': chunk[-1]['end_formatted'],
                'segment_ids': ','.join([str(seg['id']) for seg in chunk])
            }
            
            documents.append(text)
            embeddings.append(embedding)
            metadatas.append(metadata)
            ids.append(f"chunk_{i}")
        
        ## Add to collection
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
        
        print(f"Added {len(chunks)} chunks to vector store")
    
    ## This function groups the segments into larger chunks
    def _create_chunks(self, segments: List[Dict], chunk_size: int = 5) -> List[List[Dict]]:

        chunks = []
        for i in range(0, len(segments), chunk_size):
            chunk = segments[i:i + chunk_size]
            if chunk:
                chunks.append(chunk)
        return chunks
    
    ## This function searches for the relevant commentary chunks based on the users query returns the n_results number of relevant chunks
    def search(self, query: str, n_results: int = 5) -> List[Dict]:

        ## Get query embedding
        query_embedding = self.embedding_model.encode(query).tolist()
        
        ## Search
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            include=['documents', 'metadatas', 'distances']
        )
        
        ## Format results
        formatted = []
        for i in range(len(results['documents'][0])):
            formatted.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'similarity': 1 - results['distances'][0][i]  # Convert distance to similarity
            })
        
        return formatted

In [28]:
## Initialize vector store and add segments
vector_store = CommentaryVectorStore()
vector_store.add_segments(segments)

## Test search
test_results = vector_store.search("three point shot", n_results=3)
print("\nTest search for 'three point shot':")
for result in test_results:
    print(f"\n[{result['metadata']['start_formatted']} - {result['metadata']['end_formatted']}]")
    print(f"Similarity: {result['similarity']:.3f}")
    print(f"Text: {result['text'][:200]}...")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x176a51be0>: Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 7c626162-bda6-4dec-85ac-2f299527c04e)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Initialized vector store: basketball_commentary
Added 120 chunks to vector store

Test search for 'three point shot':

[22:42 - 22:53]
Similarity: 0.587
Text: That's good. This young man's had an excellent season in terms of taking some steps. But the three-pointer has been at times a struggle for him. Nice shot there, and it's a three-point game....

[06:25 - 06:32]
Similarity: 0.537
Text: Shocklock down a three. James a three-point. Way off. And Raymond bringing the rebound. The Lakers have missed nine of their first 11...

[09:49 - 10:05]
Similarity: 0.524
Text: Here comes James. Shruder, corner three. James, the offensive rebound. Quick move inside, draws the contact, shot won't go, but a foul, and he will go right back to the free throw line....


In [47]:
class BasketballCommentaryQA:

    ## Initialization of the Q&A commentary system
    def __init__(self, vector_store: CommentaryVectorStore, model: str = "llama3.2"):
        """
        Initialize the Q&A system.
        
        Args:
            vector_store: Vector store with commentary chunks
            model: Ollama model to use
        """
        self.vector_store = vector_store
        self.model = model
        
        ## System prompt for the LLM
        self.system_prompt = """You are an expert basketball analyst assistant. Your task is to analyze basketball game commentary and provide detailed insights about player performance.

IMPORTANT GUIDELINES:
1. Always reference specific timestamps when discussing events (format: MM:SS)
2. Provide structured analysis with clear sections
3. Include specific details from the commentary
4. Identify key plays, scoring moments, and performance trends
5. Be objective and analytical in your assessments

When analyzing player performance:
- Identify scoring plays and their timestamps
- Note defensive contributions
- Track momentum shifts
- Highlight key moments in the game

Format your response with:
- A summary section
- Detailed analysis with timestamps
- Key highlights

Note: All analysis is based on the commentary provided"""
    
    ## This function takes in a user prompt and returns an answer with timestamps
    def ask(self, question: str, n_context: int = 8) -> str:

        ## Retrieve relevant context
        context_results = self.vector_store.search(question, n_results=n_context)
        
        ## Build context string with timestamps
        context_parts = []
        for result in context_results:
            timestamp = f"[{result['metadata']['start_formatted']} - {result['metadata']['end_formatted']}]"
            context_parts.append(f"{timestamp}\n{result['text']}")
        
        context_string = "\n\n".join(context_parts)
        
        ## Build the prompt
        user_prompt = f"""Based on the following basketball game commentary, please answer this question:

QUESTION: {question}

COMMENTARY CONTEXT (with timestamps):
{context_string}

Please provide a detailed analysis with specific timestamp references."""
        
        ## Call Ollama
        response = ollama.chat(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        
        return response['message']['content']
    
    ## This function creates a chat that the user can interact with that answers questions with context and continues to ask questions until the user types 'quit'
    def interactive_chat(self):
        print("\n" + "="*60)
        print("Basketball Commentary Analysis System")
        print("="*60)
        print("\nAsk questions about the game. Type 'quit' to exit.\n")
        
        while True:
            question = input("\nYour question: ").strip()
            
            if question.lower() in ['quit', 'exit', 'q']:
                break
            
            if not question:
                print("Please enter a question.")
                continue
            print(f"Question: {question}")
            print("\nAnalyzing...\n")
            response = self.ask(question)
            print("\n" + "-"*60)
            print("ANALYSIS:")
            print("-"*60)
            print(response)
            print("-"*60)

In [48]:
## Initialization of Q&A system
qa_system = BasketballCommentaryQA(vector_store, model="llama3.2")

In [42]:
## Example question 1: Top scorer analysis
question1 = "Analyze the player that scored the most in this game"

print("Question:", question1)
print("\n" + "="*60)
response1 = qa_system.ask(question1)
print(response1)

Question: Analyze the player that scored the most in this game

**Summary**

The commentary highlights various player performances throughout the game, but the player who scored the most is Michael Jordan, although he is not present in the game (as hinted by the commentator), and also mentions Anthony Davis as one of the top scorers. However, based on the context provided, it seems that the commentary focuses more on the defense and plays made by players like Draymond Green, rather than showcasing their scoring abilities directly.

**Detailed Analysis**

* The player who scores the most in this game is not explicitly stated, but Michael Jordan's name is mentioned as the oldest scoring champion in the NBA in 23 years. However, his presence is only hinted at, and he does not appear to be playing in the game.
* Anthony Davis is mentioned as a key scorer, with commentary highlighting his impressive plays (14:10 - 14:23). The commentator praises Draymond Green's defense on Davis, particular

In [43]:
## Example question 2: Three-point shooting
question2 = "Who made the most three-point shots and when did they occur?"

print("Question:", question2)
print("\n" + "="*60)
response2 = qa_system.ask(question2)
print(response2)

Question: Who made the most three-point shots and when did they occur?

**Summary**

Based on the provided commentary, we can analyze the player performance, specifically focusing on who made the most three-point shots and when they occurred.

**Detailed Analysis**

From the commentary, it is evident that James made several significant plays, including scoring moments from beyond the arc.

* **Horten Tucker's straight-on three-pointer**: The commentator mentions, "Horten Tucker straight on three." This suggests that Tucker successfully converted a three-point shot. However, no timestamp was provided for this event.
* **Shruder's corner three-pointer**: At 09:49 - 10:05, the commentator states, "Here comes James. Shruder, corner three." Although not explicitly stated as a successful three-pointer by Shruder, it is implied that James' quick move inside and drawing contact resulted in Shruder scoring an uncontested shot.
* **Caruso's wide-open three-pointer**: At 12:27 - 12:42, the commen

In [44]:
## Example question 3: Key moments
question3 = "What were the key momentum shifts in the game?"

print("Question:", question3)
print("\n" + "="*60)
response3 = qa_system.ask(question3)
print(response3)

Question: What were the key momentum shifts in the game?

**Momentum Shift Analysis**

Based on the provided commentary, the following momentum shifts were observed in the game:

1. **Early Deficit**: The Lakers started strong, scoring nine of their first 11 points, but then missed several shots, including a three-point attempt by James (06:25 - 06:32). This initial surge was short-lived.

2. **Golden State Momentum Shift**: After the Lakers' early struggles, Golden State began to take control of the game with two blowout leads (24:17 - 24:35). The team's ability to capitalize on opportunities and maintain a significant lead showcased their overall strength.

3. **Defensive Stand**: The Lakers' defense played a crucial role in shifting momentum in their favor (20:05 - 20:21). A block by Green, followed by a foul, gave the Lakers two free throws, which they successfully converted, demonstrating their ability to capitalize on defensive stops.

4. **Technical Foul and Transition Issues**:

In [49]:
## Implementation of the interactive chat
qa_system.interactive_chat()


Basketball Commentary Analysis System

Ask questions about the game. Type 'quit' to exit.

Question: Who had the most rebounds this game?

Analyzing...


------------------------------------------------------------
ANALYSIS:
------------------------------------------------------------
**Summary**

Based on the commentary provided, we can analyze the rebounding performance of key players in the game. The commentary mentions several instances where different players grab rebounds, but we need to identify who had the most rebounds in the game.

**Detailed Analysis with Timestamps**

The commentary provides specific details about rebounding plays throughout the game. Here's a breakdown of each instance:

1. Raymond bringing the rebound (06:25-06:32)
	* This is an early defensive rebound, showcasing the Lakers' ability to grab important boards.
2. Davis grabbing the rebound (18:22-18:39)
	* Montrez-Harrell's spin move is unsuccessful, and Kuzma tips the ball away from him, securing another

In [46]:
## Save transcription to JSON
with open('transcription.json', 'w') as f:
    json.dump({
        'full_text': transcription['text'],
        'segments': segments
    }, f, indent=2)

print("Transcription saved to transcription.json")

## Save as readable text with timestamps
with open('transcription.txt', 'w') as f:
    for seg in segments:
        f.write(f"[{seg['start_formatted']} - {seg['end_formatted']}] {seg['text']}\n")

print("Readable transcription saved to transcription.txt")

Transcription saved to transcription.json
Readable transcription saved to transcription.txt
